In [ ]:
!pip install tensorflow scikit-image opencv-python


In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import skimage
print(f"scikit-image version: {skimage.__version__}")
from skimage.feature import graycomatrix, graycoprops
import tensorflow as tf
from tensorflow.keras import layers, Model

scikit-image version: 0.25.2


In [ ]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle datasets download -d briscdataset/brisc2025
!unzip -q brisc2025.zip


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 8

train_img_path = "/content/brisc2025/segmentation_task/train/images"
train_mask_path = "/content/brisc2025/segmentation_task/train/masks"

test_img_path = "/content/brisc2025/segmentation_task/test/images"
test_mask_path = "/content/brisc2025/segmentation_task/test/masks"


In [ ]:
train_images = sorted([os.path.join(train_img_path, f)
                       for f in os.listdir(train_img_path)])

train_masks = sorted([os.path.join(train_mask_path, f)
                      for f in os.listdir(train_mask_path)])

test_images = sorted([os.path.join(test_img_path, f)
                      for f in os.listdir(test_img_path)])

test_masks = sorted([os.path.join(test_mask_path, f)
                     for f in os.listdir(test_mask_path)])

print("Train samples:", len(train_images))
print("Test samples:", len(test_images))


In [ ]:
def load_image_mask(img_path, mask_path):

    # Load image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.keras.applications.resnet.preprocess_input(img)

    # Load mask (IMPORTANT: nearest resize)
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method='nearest')
    mask = tf.cast(mask > 0, tf.float32)

    return img, mask

In [ ]:
def augment(img, mask):

    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)

    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)

    return img, mask

In [ ]:
train_dataset_seg = tf.data.Dataset.from_tensor_slices((train_images, train_masks))
train_dataset_seg = train_dataset_seg.map(load_image_mask)
train_dataset_seg = train_dataset_seg.map(augment)
train_dataset_seg = train_dataset_seg.shuffle(1000)
train_dataset_seg = train_dataset_seg.batch(BATCH_SIZE)
train_dataset_seg = train_dataset_seg.prefetch(tf.data.AUTOTUNE)

test_dataset_seg = tf.data.Dataset.from_tensor_slices((test_images, test_masks))
test_dataset_seg = test_dataset_seg.map(load_image_mask)
test_dataset_seg = test_dataset_seg.batch(BATCH_SIZE)
test_dataset_seg = test_dataset_seg.prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_resnet_unet(input_shape):

    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )

    base_model.trainable = False  # Freeze initially

    # Encoder skip connections
    c1 = base_model.get_layer("conv1_relu").output
    c2 = base_model.get_layer("conv2_block3_out").output
    c3 = base_model.get_layer("conv3_block4_out").output
    c4 = base_model.get_layer("conv4_block6_out").output
    c5 = base_model.get_layer("conv5_block3_out").output

    # Decoder
    u1 = layers.UpSampling2D()(c5)
    u1 = layers.Concatenate()([u1, c4])
    u1 = layers.Conv2D(256, 3, padding="same", activation="relu")(u1)
    u1 = layers.Conv2D(256, 3, padding="same", activation="relu")(u1)

    u2 = layers.UpSampling2D()(u1)
    u2 = layers.Concatenate()([u2, c3])
    u2 = layers.Conv2D(128, 3, padding="same", activation="relu")(u2)
    u2 = layers.Conv2D(128, 3, padding="same", activation="relu")(u2)

    u3 = layers.UpSampling2D()(u2)
    u3 = layers.Concatenate()([u3, c2])
    u3 = layers.Conv2D(64, 3, padding="same", activation="relu")(u3)
    u3 = layers.Conv2D(64, 3, padding="same", activation="relu")(u3)

    u4 = layers.UpSampling2D()(u3)
    u4 = layers.Concatenate()([u4, c1])
    u4 = layers.Conv2D(32, 3, padding="same", activation="relu")(u4)
    u4 = layers.Conv2D(32, 3, padding="same", activation="relu")(u4)

    u5 = layers.UpSampling2D()(u4)
    outputs = layers.Conv2D(1, 1, activation="sigmoid")(u5)

    model = Model(base_model.input, outputs)

    return model

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return intersection / (union + 1e-6)

def tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3):
    smooth = 1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    TP = tf.reduce_sum(y_true_f * y_pred_f)
    FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))

    return 1 - (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)

In [ ]:
seg_model = build_resnet_unet((IMG_SIZE, IMG_SIZE, 3))

seg_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tversky_loss,
    metrics=[iou_metric]
)

seg_model.summary()

In [ ]:
history = seg_model.fit(
    train_dataset_seg,
    validation_data=test_dataset_seg,
    epochs=15
)

In [ ]:
seg_model.trainable = True

seg_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=bce_dice_loss,
    metrics=[iou_metric]
)

history_finetune = seg_model.fit(
    train_dataset_seg,
    validation_data=test_dataset_seg,
    epochs=30
)

In [ ]:
from skimage.feature import graycomatrix, graycoprops
import cv2
import numpy as np

def extract_texture_from_prediction(image, pred_mask):

    # Convert image to grayscale
    gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

    # Ensure mask is binary 0/255
    pred_mask = (pred_mask > 0).astype(np.uint8) * 255

    # Extract tumor region
    tumor_region = cv2.bitwise_and(gray, gray, mask=pred_mask)

    # Compute GLCM
    glcm = graycomatrix(
        tumor_region,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

    return contrast, correlation, energy, homogeneity

In [ ]:
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Pick random index
idx = random.randint(0, len(test_images) - 1)

# ==============================
# Load Original Image
# ==============================
img = cv2.imread(test_images[idx])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

# ==============================
# Load Ground Truth Mask
# ==============================
gt_mask = cv2.imread(test_masks[idx], cv2.IMREAD_GRAYSCALE)
gt_mask = cv2.resize(gt_mask, (IMG_SIZE, IMG_SIZE))
gt_mask = (gt_mask > 0).astype(np.uint8)

# ==============================
# ResNet Preprocessing
# ==============================
img_preprocessed = tf.keras.applications.resnet.preprocess_input(
    img_resized.astype(np.float32)
)

# ==============================
# Run Segmentation
# ==============================
input_img = np.expand_dims(img_preprocessed, axis=0)
pred = seg_model.predict(input_img)[0]

print("Prediction range:", pred.min(), pred.max())

# Try threshold tuning if needed
pred_mask = (pred > 0.6).astype(np.uint8).squeeze()

# ==============================
# Texture Feature Extraction
# IMPORTANT: Use original normalized image, NOT preprocessed
# ==============================

img_normalized = img_resized / 255.0

contrast, correlation, energy, homogeneity = extract_texture_from_prediction(
    img_normalized, pred_mask
)

print("\nTexture Features (Predicted Tumor Region)")
print(f"Contrast: {contrast:.4f}")
print(f"Correlation: {correlation:.4f}")
print(f"Energy: {energy:.4f}")
print(f"Homogeneity: {homogeneity:.4f}")

# ==============================
# Visualization
# ==============================

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Original MRI")
plt.imshow(img_resized)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Ground Truth")
plt.imshow(gt_mask, cmap='gray')
plt.axis("off")

plt.subplot(1,3,3)
plt.title(
    f"Predicted\n"
    f"C:{contrast:.2f}  Corr:{correlation:.2f}\n"
    f"E:{energy:.2f}  H:{homogeneity:.2f}"
)
plt.imshow(pred_mask, cmap='gray')
plt.axis("off")

plt.show()

In [ ]:
seg_model.save("segmentation_model.h5")

CLASSIFICATION

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

train_path = "/content/brisc2025/classification_task/train"
test_path  = "/content/brisc2025/classification_task/test"


In [ ]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    test_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_dataset.class_names
print("Classes:", class_names)


Found 5000 files belonging to 4 classes.
Found 1000 files belonging to 4 classes.
Classes: ['glioma', 'meningioma', 'no_tumor', 'pituitary']


In [ ]:
normalization_layer = layers.Rescaling(1./255)

train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset  = test_dataset.map(lambda x, y: (normalization_layer(x), y))


In [ ]:
def build_classifier(input_shape, num_classes):

    inputs = layers.Input(input_shape)

    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs)


In [ ]:
clf_model = build_classifier((IMG_SIZE, IMG_SIZE, 3), len(class_names))

clf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

clf_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = clf_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=25
)


Epoch 1/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 22s 52ms/step - accuracy: 0.5685 - loss: 1.0385 - val_accuracy: 0.7520 - val_loss: 0.6420
Epoch 2/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.7956 - loss: 0.5528 - val_accuracy: 0.7960 - val_loss: 0.4939
Epoch 3/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - accuracy: 0.8600 - loss: 0.3976 - val_accuracy: 0.8170 - val_loss: 0.4281
Epoch 4/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 39ms/step - accuracy: 0.8817 - loss: 0.3258 - val_accuracy: 0.8730 - val_loss: 0.3539
Epoch 5/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.8985 - loss: 0.2729 - val_accuracy: 0.8700 - val_loss: 0.3352
Epoch 6/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 41ms/step - accuracy: 0.9195 - loss: 0.2314 - val_accuracy: 0.9020 - val_loss: 0.2822
Epoch 7/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.9258 - loss: 0.2021 - val_accuracy: 0.9080 - val_loss: 0.2647
Epoch 8/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - accuracy: 0.9398 - loss: 0.1649 - 

In [ ]:
loss, accuracy = clf_model.evaluate(test_dataset)
print("Test Accuracy:", accuracy)


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9308 - loss: 0.3372
Test Accuracy: 0.9390000104904175


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in test_dataset:
    preds = clf_model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━

In [ ]:
import random

for images, labels in test_dataset.shuffle(100).take(1):

    preds = clf_model.predict(images)

    idx = random.randint(0, images.shape[0] - 1)

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.title("Image")
    plt.imshow(images[idx])
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title(f"Predicted: {class_names[np.argmax(preds[idx])]}")
    plt.imshow(images[idx])
    plt.axis("off")

    plt.show()

    break


In [ ]:
clf_model.save("classification_model.h5")